In [1]:
print("Pyspark")

Pyspark


# CLASE 2 - Transformaciones básicas por PySpark

## select, filter groupyBy, agg, joins, nulos, Pandas vs Spark

In [2]:
#select / withColumn ----> ¿Qué columnas y qué indicadores derivo?
#filter              ----> ¿qué filas cumplen con la regla?
#groupBy + agg       ----> ¿cuál es el resumen por dimensión?
#join                ----> ¿cómo cruzo hecho + dimensión?
#nulos y tipos       ----> pone null y nosotros decidimos que hacer

In [3]:
# Arrancar Spark: una sesion LOCAL para toda la clase.
# El worker de Python DEBE ser el mismo interprete del kernel.
# Si no se configura, Windows intenta reconectar y aparece
# "Python worker failed to connect back".

import os
import sys
from pathlib import Path

os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

RAIZ = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    DateType,
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)

activa = SparkSession.getActiveSession()
if activa is not None:
    activa.stop()

spark = (
    SparkSession.builder
    .appName("Unidad2-Clase2-Transformaciones")
    .master("local[*]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.pyspark.python", sys.executable)
    .config("spark.pyspark.driver.python", sys.executable)
    .config("spark.sql.execution.arrow.pyspark.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("  App      :", spark.sparkContext.appName)
print("  Master   :", spark.sparkContext.master)
print("  Version  :", spark.version)
print("  Python   :", spark.sparkContext.getConf().get("spark.pyspark.python"))

c:\Users\JORGE\miniconda3\envs\py_env\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


  App      : Unidad2-Clase2-Transformaciones
  Master   : local[*]
  Version  : 4.2.0
  Python   : c:\Users\JORGE\miniconda3\envs\py_env\python.exe


In [4]:
esquema_ventas = StructType(
    [
        StructField("venta_id", IntegerType(), False),
        StructField("cliente_id", IntegerType(), False),
        StructField("producto_id", IntegerType(), False),
        StructField("fecha_venta", DateType(), False),
        StructField("cantidad", IntegerType(), False),
        StructField("precio_unitario", DoubleType(), False),
        StructField("descuento", DoubleType(), False),
        StructField("total", DoubleType(), False),
        StructField("canal", StringType(), False),
    ]
)

ruta_csv = (RAIZ / "data" / "row" / "ventas.csv").resolve()
ruta_pq =  (RAIZ / "data" / "processed" / "ventas.parquet").resolve()

if ruta_pq.exists():
    ventas = spark.read.parquet(ruta_pq.as_posix())
    print("Fuente ventas: Parquet")

else:
    ventas = (
        spark.read
        .option("header", "true")
        .schema(esquema_ventas)
        .csv(ruta_csv.as_posix())
    )
    print("Fuente ventas: CSV (no había parquet)")
    
    
clientes = (
    spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv((RAIZ / "data" / "row" / "clientes.csv").resolve().as_posix())
)

productos = (
    spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv((RAIZ / "data" / "row" / "productos.csv").resolve().as_posix())
)

print("ventas       :", ventas.count(),    "filas | ", ventas.columns)
print("clientes     :", clientes.count(),  "filas | ", clientes.columns)
print("productos    :", productos.count(), "filas | ", productos.columns)

ventas.printSchema()

Fuente ventas: Parquet
ventas       : 200000 filas |  ['venta_id', 'cliente_id', 'producto_id', 'fecha_venta', 'cantidad', 'precio_unitario', 'descuento', 'total', 'canal']
clientes     : 10000 filas |  ['cliente_id', 'nombre', 'email', 'ciudad', 'pais', 'segmento', 'fecha_registro']
productos    : 2000 filas |  ['producto_id', 'nombre', 'categoria', 'precio', 'coste', 'stock', 'activo']
root
 |-- venta_id: integer (nullable = true)
 |-- cliente_id: integer (nullable = true)
 |-- producto_id: integer (nullable = true)
 |-- fecha_venta: date (nullable = true)
 |-- cantidad: integer (nullable = true)
 |-- precio_unitario: double (nullable = true)
 |-- descuento: double (nullable = true)
 |-- total: double (nullable = true)
 |-- canal: string (nullable = true)



## select y withColumn

In [5]:
#select / withColumn ----> ¿Qué columnas y qué indicadores derivo?
#filter              ----> ¿qué filas cumplen con la regla?
#groupBy + agg       ----> ¿cuál es el resumen por dimensión?
#join                ----> ¿cómo cruzo hecho + dimensión?
#nulos y tipos       ----> pone null y nosotros decidimos que hacer

#Victor requiero un informe de tickets.
#Para un informe de tickets, ¿necesito precio_unitario y descuento?

solo_informe = ventas.select(
    "venta_id",
    "cliente_id",
    "fecha_venta",
    "canal",
    "total",
)

print("Columnas originales:", ventas.columns)
print("Columnas del select:", solo_informe.columns)
print("¿Se ve el recorte?")
solo_informe.show()

Columnas originales: ['venta_id', 'cliente_id', 'producto_id', 'fecha_venta', 'cantidad', 'precio_unitario', 'descuento', 'total', 'canal']
Columnas del select: ['venta_id', 'cliente_id', 'fecha_venta', 'canal', 'total']
¿Se ve el recorte?
+--------+----------+-----------+------+-------+
|venta_id|cliente_id|fecha_venta| canal|  total|
+--------+----------+-----------+------+-------+
|       1|      2893| 2025-08-04|   web|1563.57|
|       2|      1187| 2025-09-03|   app|3251.04|
|       3|      1053| 2025-11-02|   web| 269.46|
|       4|        86| 2024-03-29|   web| 829.03|
|       5|      9774| 2022-10-11|   web| 460.57|
|       6|      6472| 2023-10-04|   app| 182.29|
|       7|      3167| 2024-11-30|   web| 688.44|
|       8|      5727| 2025-02-10|   web| 364.45|
|       9|      6115| 2025-06-01|   web|  30.24|
|      10|      1805| 2025-08-10|   web|  65.54|
|      11|      2865| 2024-03-26|   web| 592.68|
|      12|      5423| 2025-06-15|   app|   3.08|
|      13|      8366| 202

In [6]:
# Withcolumn
# Julio: "Requiero que me entregues el año, mes, un flag de ticket gande y un alias legible."

ventas_enriquecidas = (
    ventas
    .withColumn("anio", F.year("fecha_venta"))
    .withColumn("mes", F.month("fecha_venta"))
    .withColumn("ticket_grande", F.col("total") >= 120)
    .withColumn("importe", F.round(F.col("total"), 2))
    .drop("cantidad")
)

print("¿Quedaron año, mes, ticket grande, importe y ya no está cantidad?")
print("Columnas:", ventas_enriquecidas.columns)
ventas_enriquecidas.select(
    "venta_id", "fecha_venta", "anio", "mes", "total", "importe", "ticket_grande" , "canal" 
).show(8, truncate=False)


print("¿Que valores unicos tiene canal?")
ventas.select("canal").distinct().orderBy("canal").show()


¿Quedaron año, mes, ticket grande, importe y ya no está cantidad?
Columnas: ['venta_id', 'cliente_id', 'producto_id', 'fecha_venta', 'precio_unitario', 'descuento', 'total', 'canal', 'anio', 'mes', 'ticket_grande', 'importe']
+--------+-----------+----+---+-------+-------+-------------+-----+
|venta_id|fecha_venta|anio|mes|total  |importe|ticket_grande|canal|
+--------+-----------+----+---+-------+-------+-------------+-----+
|1       |2025-08-04 |2025|8  |1563.57|1563.57|true         |web  |
|2       |2025-09-03 |2025|9  |3251.04|3251.04|true         |app  |
|3       |2025-11-02 |2025|11 |269.46 |269.46 |true         |web  |
|4       |2024-03-29 |2024|3  |829.03 |829.03 |true         |web  |
|5       |2022-10-11 |2022|10 |460.57 |460.57 |true         |web  |
|6       |2023-10-04 |2023|10 |182.29 |182.29 |true         |app  |
|7       |2024-11-30 |2024|11 |688.44 |688.44 |true         |web  |
|8       |2025-02-10 |2025|2  |364.45 |364.45 |true         |web  |
+--------+-----------+----

## filter / where

In [7]:
#Y = &
#O = |
#NO = ~

# ¿Cuantas ventas WEB de 500 euros o más hubo?
web_grandes = ventas.filter(
    (F.col("canal") == "web") & (F.col("total") >= 500)
)

print("¿Cuantas ventas web superan los 500 euros")
print("Respuesta:", web_grandes.count()) #1 job
web_grandes.select("venta_id", "fecha_venta", "total", "canal").show(5)

#¿ Qué Ventas de app o tienda con total entre 50 y 80 euros hay?
rango_app_tienda = ventas.filter(
    F.col("canal").isin("app", "tienda") & F.col("total").between(50, 80)
)

print("Cuantas ventas de app/tienda tienen tickets de 50 a 80?")
print("Repsuesta:", rango_app_tienda.count())

#¿Qué pasa con lo que no es web?
no_web = ventas.filter(~(F.col("canal") == "web"))
print("¿Cuantas ventas no son web?")
print("Respuesta:", no_web.count())


¿Cuantas ventas web superan los 500 euros
Respuesta: 19625
+--------+-----------+-------+-----+
|venta_id|fecha_venta|  total|canal|
+--------+-----------+-------+-----+
|       1| 2025-08-04|1563.57|  web|
|       4| 2024-03-29| 829.03|  web|
|       7| 2024-11-30| 688.44|  web|
|      11| 2024-03-26| 592.68|  web|
|      14| 2025-10-06| 925.98|  web|
+--------+-----------+-------+-----+
only showing top 5 rows
Cuantas ventas de app/tienda tienen tickets de 50 a 80?
Repsuesta: 11337
¿Cuantas ventas no son web?
Respuesta: 110034


## GroupBy + Agg


In [8]:
# Pregunta: Por cada canal: ¿Cuántas lineas, cuánta facturación, que ticket medio y cuantos clientes distintos?

por_canal = (
    ventas
    .groupBy("canal")
    .agg(
        F.count("*").alias("num_ventas"),
        F.round(F.sum("total"), 2).alias("facturacion"),
        F.round(F.avg("total"), 2).alias("ticket medio"),
        F.countDistinct("cliente_id").alias("clientes_distintos"),
        F.round(F.max("total"), 2).alias("ticket_max"),
        F.round(F.min("total"), 2).alias("ticket_min"),
    )
    .orderBy(F.col("facturacion").desc())
)

print("¿Qué canal factura más y con qué ticket medio?")
print("La tabla (1 job al show)")
por_canal.show()

¿Qué canal factura más y con qué ticket medio?
La tabla (1 job al show)
+--------+----------+-------------+------------+------------------+----------+----------+
|   canal|num_ventas|  facturacion|ticket medio|clientes_distintos|ticket_max|ticket_min|
+--------+----------+-------------+------------+------------------+----------+----------+
|     web|     89966|3.793579302E7|      421.67|              9972|   14986.9|      0.71|
|     app|     60210|2.516147375E7|       417.9|              9830|   14986.9|      0.71|
|  tienda|     39859|1.706546291E7|      428.15|              9429|   14793.0|      0.71|
|telefono|      9965|   4088686.05|       410.3|              5779|   14442.3|      0.94|
+--------+----------+-------------+------------+------------------+----------+----------+



## JOINs

In [9]:
#inner
#left ---------------------------> Es el más usado en analíticas
#right
#outer

#Producto cartesiano

#cliente_id = 1
#cliente_id = 80
#cliente_id = 200
#cliente_id = 200

In [10]:
# Pregunta: "Si dibujo las tablas en la pizarra, ¿Que sale con cada show?"

ventas_demo = spark.createDataFrame(
    [
        (1, 10, 50.0),
        (2, 10, 80.0),
        (3, 20, 30.0),
        (4, 99, 99.0),
    ],
    schema=["venta_id", "cliente_id", "total"],
)

print("Ventas demo (4) y clientes demo(3). cliente 99 no existe; Luis no compró")
ventas_demo.show()

Ventas demo (4) y clientes demo(3). cliente 99 no existe; Luis no compró
+--------+----------+-----+
|venta_id|cliente_id|total|
+--------+----------+-----+
|       1|        10| 50.0|
|       2|        10| 80.0|
|       3|        20| 30.0|
|       4|        99| 99.0|
+--------+----------+-----+



In [11]:
clientes_demo = spark.createDataFrame(
    [
        (10, "Ana", "Colombia"),
        (20, "Victor", "Colombia"),
        (30, "Luis", "Mexico"),
    ],
    schema=["cliente_id", "nombre", "pais"],
)

print("Ventas demo (4) y clientes demo(3). cliente 99 no existe; Luis no compró")
ventas_demo.show()
clientes_demo.show()

Ventas demo (4) y clientes demo(3). cliente 99 no existe; Luis no compró
+--------+----------+-----+
|venta_id|cliente_id|total|
+--------+----------+-----+
|       1|        10| 50.0|
|       2|        10| 80.0|
|       3|        20| 30.0|
|       4|        99| 99.0|
+--------+----------+-----+

+----------+------+--------+
|cliente_id|nombre|    pais|
+----------+------+--------+
|        10|   Ana|Colombia|
|        20|Victor|Colombia|
|        30|  Luis|  Mexico|
+----------+------+--------+



In [13]:
# Inner: Intersección.

#Pregunta: ¿Qué ventas peudo reportar con nombe, porque el cliente existe?

inner = (
    ventas_demo.join(clientes_demo, on="cliente_id", how="inner")
    .select("venta_id", "cliente_id", "nombre", "pais", "total")
    .orderBy("venta_id")
)

print("INNER - filas:", inner.count())
inner.show()

INNER - filas: 3
+--------+----------+------+--------+-----+
|venta_id|cliente_id|nombre|    pais|total|
+--------+----------+------+--------+-----+
|       1|        10|   Ana|Colombia| 50.0|
|       2|        10|   Ana|Colombia| 80.0|
|       3|        20|Victor|Colombia| 30.0|
+--------+----------+------+--------+-----+



In [15]:
#left

#¿Quiero ver todas la ventas y marcar las que no tienen lciente?

left = (
    ventas_demo.join(clientes_demo, on="cliente_id", how="left")
    .select("venta_id", "cliente_id", "nombre", "pais", "total")
    .orderBy("venta_id")
)

print("LEFT - filas:", left.count())
left.show()

huerfanas = left.filter(F.col("nombre").isNull())
print("Huerfanas (left + isNull)")
huerfanas.show()

LEFT - filas: 4
+--------+----------+------+--------+-----+
|venta_id|cliente_id|nombre|    pais|total|
+--------+----------+------+--------+-----+
|       1|        10|   Ana|Colombia| 50.0|
|       2|        10|   Ana|Colombia| 80.0|
|       3|        20|Victor|Colombia| 30.0|
|       4|        99|  NULL|    NULL| 99.0|
+--------+----------+------+--------+-----+

Huerfanas (left + isNull)
+--------+----------+------+----+-----+
|venta_id|cliente_id|nombre|pais|total|
+--------+----------+------+----+-----+
|       4|        99|  NULL|NULL| 99.0|
+--------+----------+------+----+-----+



## Ejercicios (los resuelves tú)

Usa `ventas`, `clientes` y `productos` (ya están en memoria). Encadena **transformaciones** y termina con una **acción** (`show`, `count`, `head`). No hagas `collect()` de las 200.000 filas.

Regla: la respuesta es un número o una tablita corta, no un volcado.

| # | Pregunta de negocio | Qué deberías usar |
|---|---|---|
| 1 | ¿Cuántas ventas de **2025** se hicieron por **teléfono**? | `filter` + `count` |
| 2 | Crea `ingreso_lista = cantidad * precio_unitario` y muestra 8 filas junto a `total` y `descuento`. ¿Por qué a veces no coinciden? | `withColumn` + `select` + `show` |
| 3 | De las ventas **web** con `total >= 1000`, enseña 10 (id, fecha, total). | `filter` + `select` + `show` |
| 4 | Por **canal**: facturación y ticket medio **solo en 2024**. ¿Sigue ganando web? | `filter` + `groupBy` + `agg` |
| 5 | ¿Qué **categoría** factura más? (cruzar ventas con productos) | `join` + `groupBy` + `orderBy` |
| 6 | Top **10 clientes** por gasto: nombre, segmento, país y gasto total. | `join` + `groupBy` + `orderBy` + `limit` |
| 7 | ¿Cuántos **productos del catálogo nunca se vendieron**? | `join` left + `isNull` + `count` |
| 8 | En **app**, ¿qué porcentaje de líneas lleva descuento > 0? | dos `count` (piensa el cociente) |
| 9 | En **2024**, ¿qué mes tuvo el **ticket medio** más alto? | `year`/`month` + `groupBy` |
| 10 | Clientes **platino** que compraron en **tienda**: ¿cuántos hay y cuánto gastaron? | `join` + `filter` + `agg` |

Pistas: `&` `\|` `~` en `filter`; `F.round`, `F.countDistinct`; el `JOIN` va **antes** de agregar.

In [ ]:
# Ejercicio 1 — ¿Cuántas ventas de 2025 se hicieron por teléfono?
# Pista: filter (año + canal) y luego count.



In [ ]:
# Ejercicio 2 — ingreso_lista = cantidad * precio_unitario
# Muestra 8 filas con total y descuento. ¿Por qué no siempre coinciden?



In [ ]:
# Ejercicio 3 — 10 ventas web con total >= 1000
# select de venta_id, fecha_venta, total. Acción: show(10).



In [ ]:
# Ejercicio 4 — Por canal, solo 2024: facturación y ticket medio.
# ¿Sigue ganando web?



In [ ]:
# Ejercicio 5 — ¿Qué categoría factura más?
# JOIN ventas-productos, groupBy categoria, orderBy desc.



In [ ]:
# Ejercicio 6 — Top 10 clientes por gasto
# nombre, segmento, país, gasto_total. JOIN + groupBy + limit(10).



In [ ]:
# Ejercicio 7 — Productos del catálogo que nunca se vendieron
# LEFT join desde productos. Cuenta los que quedan con venta_id nulo.



In [ ]:
# Ejercicio 8 — En canal app, ¿qué % de líneas tiene descuento > 0?
# Dos count (con y sin filtro). El cociente es la acción en Python.



In [ ]:
# Ejercicio 9 — En 2024, ¿qué mes tuvo el ticket medio más alto?
# withColumn mes + filter año + groupBy + orderBy.



In [ ]:
# Ejercicio 10 — Clientes platino que compraron en tienda
# ¿Cuántos clientes distintos y cuánto gastaron en total?

